In [1]:
import pandas as pd

In [2]:
assignments = pd.read_csv("assignments.csv", sep=";")
caregivers = pd.read_csv("caregivers.csv", sep=";")
clients = pd.read_csv("clients.csv", sep=";")

In [3]:
print(len(caregivers))

print(len(caregivers[caregivers["id"].isin(assignments["assignee"])]))

9271
7148


In [4]:
from dataclasses import dataclass
from typing import Optional, Union

import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

@dataclass
class CutConfig:
    """
    Defines the point in a caregiver timeline at which a snapshot
    is created.

    Supported modes
    ---------------

    "random_days"
        Recommended for the main predictive experiment.
        Draws one landmark uniformly between min_days and max_days
        after recruitment. The draw does NOT use the future exit date.

        Example:
            CutConfig(
                mode="random_days",
                min_days=180,
                max_days=730,
                random_state=42
            )

    "days"
        Fixed landmark measured in days after recruitment.

        Example:
            CutConfig(mode="days", value=365)

    "date"
        Fixed calendar date.

        Example:
            CutConfig(mode="date", value="2024-01-01")

    "ratio"
        Retrospective analysis only.
        Uses a fraction of recruitment -> observed_end and therefore
        depends on the future observed end date. Do NOT use this mode
        for the main prospective forecasting experiment.

        Example:
            CutConfig(mode="ratio", value=0.70)

    "full"
        Retrospective analysis only.
        Uses the full observable history up to observed_end.
    """

    mode: str = "random_days"

    # Used by "ratio", "days", and "date".
    value: Optional[Union[float, int, str]] = None

    # Used by "random_days".
    min_days: int = 180
    max_days: int = 730

    # Reproducibility of random landmarks.
    random_state: int = 42


# ============================================================
# Date parsing
# ============================================================

def _parse_caregiver_date(series: pd.Series) -> pd.Series:
    """
    Parses dates such as:
        20190903
        20190903.0
        "20190903"
    """
    cleaned = (
        series
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .replace({
            "nan": np.nan,
            "None": np.nan,
            "": np.nan
        })
    )

    return pd.to_datetime(
        cleaned,
        format="%Y%m%d",
        errors="coerce"
    )


def _parse_assignment_date(series: pd.Series) -> pd.Series:
    """
    Parses assignment dates such as:
        20160914000000
    """
    cleaned = (
        series
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .replace({
            "nan": np.nan,
            "None": np.nan,
            "": np.nan
        })
    )

    return pd.to_datetime(
        cleaned,
        format="%Y%m%d%H%M%S",
        errors="coerce"
    )


# ============================================================
# Interval utilities
# ============================================================

def _union_days(intervals):
    """
    Calculates unique covered calendar days.

    This prevents overlapping assignments from counting the
    same workday twice.
    """

    if not intervals:
        return 0

    intervals = sorted(intervals, key=lambda x: x[0])

    merged = []

    current_start, current_end = intervals[0]

    for start, end in intervals[1:]:

        if start <= current_end + pd.Timedelta(days=1):
            current_end = max(current_end, end)

        else:
            merged.append((current_start, current_end))
            current_start, current_end = start, end

    merged.append((current_start, current_end))

    return sum(
        (end - start).days + 1
        for start, end in merged
    )


# ============================================================
# Cutoff calculation
# ============================================================

def _calculate_cutoff(
    recruitment_date,
    observed_end,
    cut: CutConfig,
    rng=None
):
    """
    Calculate a snapshot cutoff.

    Important:
        - random_days, days and date do NOT require knowledge of
          the future exit date to determine the landmark.
        - ratio and full are retrospective modes and should not be
          used as the main forecasting design.
    """

    if cut.mode == "ratio":

        if cut.value is None:
            raise ValueError(
                "For ratio mode, 'value' must be provided."
            )

        ratio = float(cut.value)

        if not 0 < ratio < 1:
            raise ValueError(
                "For ratio mode, value must be > 0 and < 1."
            )

        total_days = (
            observed_end - recruitment_date
        ).days

        return (
            recruitment_date
            + pd.Timedelta(
                days=int(total_days * ratio)
            )
        )

    elif cut.mode == "days":

        if cut.value is None:
            raise ValueError(
                "For days mode, 'value' must be provided."
            )

        days = int(cut.value)

        if days < 0:
            raise ValueError(
                "For days mode, value must be >= 0."
            )

        return (
            recruitment_date
            + pd.Timedelta(days=days)
        )

    elif cut.mode == "random_days":

        if cut.min_days < 0:
            raise ValueError(
                "min_days must be >= 0."
            )

        if cut.max_days < cut.min_days:
            raise ValueError(
                "max_days must be >= min_days."
            )

        if rng is None:
            rng = np.random.default_rng(
                cut.random_state
            )

        landmark_days = int(
            rng.integers(
                low=cut.min_days,
                high=cut.max_days + 1
            )
        )

        return (
            recruitment_date
            + pd.Timedelta(
                days=landmark_days
            )
        )

    elif cut.mode == "date":

        if cut.value is None:
            raise ValueError(
                "For date mode, 'value' must be provided."
            )

        return pd.Timestamp(
            cut.value
        ).normalize()

    elif cut.mode == "full":

        return observed_end

    else:

        raise ValueError(
            "Unknown cut mode "
            f"'{cut.mode}'. "
            "Supported modes are: "
            "'random_days', 'days', 'date', "
            "'ratio', and 'full'."
        )

def _safe_mean(values):
    values = list(values)
    return float(np.mean(values)) if len(values) else np.nan


def _safe_std(values):
    values = list(values)
    return float(np.std(values, ddof=1)) if len(values) > 1 else 0.0


def _linear_trend(values):
    """
    Simple slope over an ordered sequence.
    Positive = increasing over time, negative = decreasing.
    """
    values = np.asarray(list(values), dtype=float)

    if len(values) < 2:
        return 0.0

    x = np.arange(len(values), dtype=float)

    if np.allclose(values, values[0]):
        return 0.0

    slope = np.polyfit(x, values, 1)[0]
    return float(slope)


def _calculate_recent_window_features(
    cg_assignments: pd.DataFrame,
    cutoff: pd.Timestamp,
    recruitment_date: pd.Timestamp,
    windows=(90, 180, 365),
):
    """
    Calculate recent activity features from assignments visible at the cutoff.

    Each window only uses data available before the snapshot date.
    """
    result = {}

    for window_days in windows:

        window_start = max(
            recruitment_date,
            cutoff - pd.Timedelta(days=int(window_days) - 1)
        )

        window_assignments = cg_assignments[
            (cg_assignments["_visible_end"] >= window_start)
            &
            (cg_assignments["_visible_start"] <= cutoff)
        ].copy()

        clipped_intervals = []

        for _, a in window_assignments.iterrows():
            start = max(a["_visible_start"], window_start)
            end = min(a["_visible_end"], cutoff)

            if end >= start:
                clipped_intervals.append((start, end))

        worked = _union_days(clipped_intervals)

        available_days = (
            cutoff - window_start
        ).days + 1

        off = max(
            available_days - worked,
            0
        )

        result[f"workdays_last_{window_days}d"] = worked
        result[f"offworkdays_last_{window_days}d"] = off

        result[f"work_ratio_last_{window_days}d"] = (
            worked / available_days
            if available_days > 0
            else np.nan
        )

        result[f"assignments_last_{window_days}d"] = int(
            len(window_assignments)
        )

    return result


# ============================================================
# Main function
# ============================================================

def build_caregiver_snapshot_dataset(
    caregivers: pd.DataFrame,
    assignments: pd.DataFrame,
    clients: pd.DataFrame,

    cut: Optional[CutConfig] = None,

    study_end: Optional[str] = None,

    caregiver_id_col: str = "id",
    assignment_caregiver_col: str = "assignee",
    assignment_client_col: str = "clients",
    client_id_col: str = "id",

    recruitment_col: str = "eingestellt-am",
    exit_col: str = "ausgesch-am",
    caregiver_birth_col: str = "dateOfBirth",

    assignment_start_col: str = "startDate",
    assignment_end_col: str = "endDate",

    client_class_col: str = "cluster",

    package_prefix: str = "paket_",

    # Optional caregiver-skill/client-requirement matching.
    # Mapping format:
    # {
    #   "beginnende-demenz": "erfahrung-demenz-beginnend",
    #   "fortgeschrittene-demenz-alzheimer": "erfahrung-alzheimer",
    #   ...
    # }
    requirement_experience_map: Optional[dict] = None,

    recent_windows=(90, 180, 365),

) -> pd.DataFrame:

    """
    Creates one flattened snapshot row per caregiver.

    Features are generated ONLY from assignments visible up to
    the configured cutoff date.

    Returns:
        caregiver master data
        + snapshot metadata
        + survival target
        + work-history features
        + package exposure
        + client-class exposure
    """

    caregivers = caregivers.copy()
    assignments = assignments.copy()
    clients = clients.copy()

    # --------------------------------------------------------
    # Cutoff configuration / reproducible RNG
    # --------------------------------------------------------

    if cut is None:
        cut = CutConfig()

    rng = np.random.default_rng(
        cut.random_state
    )

    # --------------------------------------------------------
    # IDs as strings
    # --------------------------------------------------------

    caregivers[caregiver_id_col] = (
        caregivers[caregiver_id_col]
        .astype(str)
        .str.strip()
    )

    assignments[assignment_caregiver_col] = (
        assignments[assignment_caregiver_col]
        .astype(str)
        .str.strip()
    )

    assignments[assignment_client_col] = (
        assignments[assignment_client_col]
        .astype(str)
        .str.strip()
    )

    clients[client_id_col] = (
        clients[client_id_col]
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # Dates
    # --------------------------------------------------------

    caregivers["_recruitment_date"] = (
        _parse_caregiver_date(
            caregivers[recruitment_col]
        )
    )

    caregivers["_exit_date"] = (
        _parse_caregiver_date(
            caregivers[exit_col]
        )
    )

    if caregiver_birth_col in caregivers.columns:
        caregivers["_birth_date"] = (
            _parse_caregiver_date(
                caregivers[caregiver_birth_col]
            )
        )

    assignments["_assignment_start"] = (
        _parse_assignment_date(
            assignments[assignment_start_col]
        )
    )

    assignments["_assignment_end"] = (
        _parse_assignment_date(
            assignments[assignment_end_col]
        )
    )

    # Normalize to calendar days
    assignments["_assignment_start"] = (
        assignments["_assignment_start"].dt.normalize()
    )

    assignments["_assignment_end"] = (
        assignments["_assignment_end"].dt.normalize()
    )

    # --------------------------------------------------------
    # Study end
    # --------------------------------------------------------

    if study_end is None:

        possible_dates = [
            caregivers["_exit_date"].max(),
            assignments["_assignment_end"].max()
        ]

        study_end_date = max(
            d for d in possible_dates
            if pd.notna(d)
        )

    else:

        study_end_date = pd.Timestamp(study_end)

    # --------------------------------------------------------
    # Add client class to assignments
    # --------------------------------------------------------

    if requirement_experience_map is None:
        requirement_experience_map = {}

    requirement_columns = [
        c
        for c in requirement_experience_map.keys()
        if c in clients.columns
    ]

    client_lookup_columns = [
        client_id_col,
        client_class_col,
        *requirement_columns
    ]

    # Preserve order while removing duplicates
    client_lookup_columns = list(
        dict.fromkeys(client_lookup_columns)
    )

    client_lookup = clients[
        client_lookup_columns
    ].drop_duplicates(
        subset=[client_id_col]
    )

    assignments = assignments.merge(
        client_lookup,
        left_on=assignment_client_col,
        right_on=client_id_col,
        how="left",
        suffixes=("", "_client")
    )

    # --------------------------------------------------------
    # Package columns
    # --------------------------------------------------------

    package_columns = [
        c for c in assignments.columns
        if c.startswith(package_prefix)
    ]

    # --------------------------------------------------------
    # Possible client classes
    # --------------------------------------------------------

    client_classes = sorted(
        clients[client_class_col]
        .dropna()
        .unique()
    )

    result_rows = []

    # ========================================================
    # One caregiver at a time
    # ========================================================

    for _, caregiver in caregivers.iterrows():

        caregiver_id = caregiver[caregiver_id_col]

        recruitment_date = caregiver["_recruitment_date"]
        exit_date = caregiver["_exit_date"]

        # Cannot construct survival record
        if pd.isna(recruitment_date):
            continue

        # ----------------------------------------------------
        # Event / censoring
        # ----------------------------------------------------

        if (
            pd.notna(exit_date)
            and exit_date <= study_end_date
        ):

            observed_end = exit_date
            event = 1

        else:

            observed_end = study_end_date
            event = 0

        # Invalid chronology
        if observed_end < recruitment_date:
            continue

        # ----------------------------------------------------
        # Snapshot cutoff
        # ----------------------------------------------------

        cutoff = _calculate_cutoff(
            recruitment_date,
            observed_end,
            cut,
            rng=rng
        )

        # The snapshot must occur while the caregiver is still
        # observable. For prospective landmark modes we NEVER
        # move a cutoff backwards to the known exit date, because
        # doing so would reintroduce target leakage.
        if cutoff < recruitment_date:
            continue

        if cut.mode in {
            "random_days",
            "days",
            "date"
        }:
            if cutoff >= observed_end:
                continue

        # ratio/full are retained only for retrospective analyses.
        # They explicitly depend on observed_end.

        # ----------------------------------------------------
        # Snapshot survival targets
        # ----------------------------------------------------

        days_since_recruitment = (
            cutoff - recruitment_date
        ).days

        remaining_observed_days = (
            observed_end - cutoff
        ).days

        # Event after snapshot
        event_after_snapshot = int(
            event == 1 and exit_date > cutoff
        )

        # ----------------------------------------------------
        # Caregiver assignments
        # ----------------------------------------------------

        cg_assignments = assignments[
            assignments[
                assignment_caregiver_col
            ] == caregiver_id
        ].copy()

        # Only assignments which have started before snapshot
        cg_assignments = cg_assignments[
            cg_assignments["_assignment_start"] <= cutoff
        ].copy()

        # Ignore assignments ending before recruitment
        cg_assignments = cg_assignments[
            cg_assignments["_assignment_end"] >= recruitment_date
        ].copy()

        # ----------------------------------------------------
        # Clip assignments to observable snapshot history
        # ----------------------------------------------------

        if not cg_assignments.empty:

            cg_assignments["_visible_start"] = (
                cg_assignments["_assignment_start"]
                .clip(lower=recruitment_date)
            )

            cg_assignments["_visible_end"] = (
                cg_assignments["_assignment_end"]
                .clip(upper=cutoff)
            )

            cg_assignments = cg_assignments[
                cg_assignments["_visible_end"]
                >=
                cg_assignments["_visible_start"]
            ].copy()

            cg_assignments["_visible_days"] = (
                (
                    cg_assignments["_visible_end"]
                    -
                    cg_assignments["_visible_start"]
                ).dt.days
                + 1
            )

        # ----------------------------------------------------
        # Start row with caregiver master data
        # ----------------------------------------------------

        row = caregiver.drop(
            labels=[
                "_recruitment_date",
                "_exit_date",
                "_birth_date"
            ],
            errors="ignore"
        ).to_dict()

        # ----------------------------------------------------
        # Snapshot metadata
        # ----------------------------------------------------

        row.update({

            "snapshot_date":
                cutoff,

            "recruitment_date_clean":
                recruitment_date,

            "exit_date_clean":
                exit_date,

            "study_end":
                study_end_date,

            "snapshot_ratio":
                cut.value
                if cut.mode == "ratio"
                else np.nan,

            "snapshot_mode":
                cut.mode,

            "snapshot_landmark_days":
                days_since_recruitment,

            "snapshot_random_min_days":
                cut.min_days
                if cut.mode == "random_days"
                else np.nan,

            "snapshot_random_max_days":
                cut.max_days
                if cut.mode == "random_days"
                else np.nan,

            "snapshot_random_state":
                cut.random_state
                if cut.mode == "random_days"
                else np.nan,

            "days_since_recruitment":
                days_since_recruitment,

            # TARGET / SURVIVAL VARIABLES
            "target_event":
                event_after_snapshot,

            "target_remaining_days":
                remaining_observed_days,

            "target_observed_end":
                observed_end

        })

        # ----------------------------------------------------
        # Age features
        # ----------------------------------------------------

        if (
            "_birth_date" in caregiver.index
            and
            pd.notna(caregiver["_birth_date"])
        ):

            birth = caregiver["_birth_date"]

            row["age_at_recruitment"] = (
                recruitment_date - birth
            ).days / 365.25

            row["age_at_snapshot"] = (
                cutoff - birth
            ).days / 365.25

        # ====================================================
        # No assignments yet
        # ====================================================

        if cg_assignments.empty:

            row.update({

                "number_assignments": 0,

                "number_unique_clients": 0,

                "workdays": 0,

                "offworkdays":
                    days_since_recruitment + 1,

                "work_ratio": 0.0,

                "mean_assignment_length": 0.0,

                "median_assignment_length": 0.0,

                "min_assignment_length": 0.0,

                "max_assignment_length": 0.0,

                "std_assignment_length": 0.0,

                "mean_offwork_length": np.nan,

                "days_since_last_assignment": np.nan,

                "repeat_client_ratio": 0.0,

                # Temporal dynamics
                "last_assignment_length": np.nan,
                "mean_assignment_length_last3": np.nan,
                "std_assignment_length_last3": 0.0,
                "assignment_length_trend": 0.0,
                "last_assignment_vs_historical_mean": np.nan,
                "last_offwork_length": 0.0,
                "mean_offwork_length_last3": 0.0,
                "std_offwork_length_last3": 0.0,
                "offwork_gap_trend": 0.0,
                "last_gap_vs_historical_mean": np.nan,
                "current_gap_ratio_to_historical": np.nan,

                # Caregiver-client fit
                "requirement_exposure_days": 0.0,
                "matched_requirement_days": 0.0,
                "unmatched_requirement_days": 0.0,
                "caregiver_client_match_ratio": np.nan,
                "caregiver_client_mismatch_ratio": np.nan,
                "requirement_type_match_ratio": np.nan,
                "requirement_types_seen": 0,
                "requirement_types_matched": 0

            })

            for window_days in recent_windows:
                row[f"workdays_last_{window_days}d"] = 0
                row[f"offworkdays_last_{window_days}d"] = (
                    min(days_since_recruitment + 1, int(window_days))
                )
                row[f"work_ratio_last_{window_days}d"] = 0.0
                row[f"assignments_last_{window_days}d"] = 0

            for cls in client_classes:

                row[f"days_client_class_{cls}"] = 0

                row[f"share_client_class_{cls}"] = 0.0

            for package in package_columns:

                safe_name = (
                    package
                    .replace("paket_", "")
                    .replace(" ", "_")
                    .replace("-", "_")
                )

                row[f"days_package_{safe_name}"] = 0

                row[f"share_package_{safe_name}"] = 0.0

            result_rows.append(row)

            continue

        # ====================================================
        # Work history
        # ====================================================

        intervals = list(
            zip(
                cg_assignments["_visible_start"],
                cg_assignments["_visible_end"]
            )
        )

        workdays = _union_days(intervals)

        total_observed_days = (
            cutoff - recruitment_date
        ).days + 1

        offworkdays = max(
            total_observed_days - workdays,
            0
        )

        lengths = cg_assignments["_visible_days"]

        row["number_assignments"] = len(
            cg_assignments
        )

        row["number_unique_clients"] = (
            cg_assignments[
                assignment_client_col
            ].nunique()
        )

        row["workdays"] = workdays

        row["offworkdays"] = offworkdays

        row["work_ratio"] = (
            workdays / total_observed_days
            if total_observed_days > 0
            else np.nan
        )

        row["mean_assignment_length"] = (
            lengths.mean()
        )

        row["median_assignment_length"] = (
            lengths.median()
        )

        row["min_assignment_length"] = (
            lengths.min()
        )

        row["max_assignment_length"] = (
            lengths.max()
        )

        row["std_assignment_length"] = (
            lengths.std()
        )

        # ----------------------------------------------------
        # Assignment gaps
        # ----------------------------------------------------

        ordered = cg_assignments.sort_values(
            "_visible_start"
        )

        gaps = []

        previous_end = None

        for _, assignment in ordered.iterrows():

            start = assignment["_visible_start"]
            end = assignment["_visible_end"]

            if previous_end is not None:

                gap = (
                    start - previous_end
                ).days - 1

                if gap >= 0:
                    gaps.append(gap)

            previous_end = max(
                previous_end,
                end
            ) if previous_end is not None else end

        row["mean_offwork_length"] = (
            np.mean(gaps)
            if gaps
            else 0.0
        )

        row["median_offwork_length"] = (
            np.median(gaps)
            if gaps
            else 0.0
        )

        row["max_offwork_length"] = (
            max(gaps)
            if gaps
            else 0.0
        )

        last_assignment_end = (
            ordered["_visible_end"].max()
        )

        row["days_since_last_assignment"] = max(
            (cutoff - last_assignment_end).days,
            0
        )

        # ====================================================
        # Temporal dynamics
        # ====================================================

        # Assignment-length dynamics
        ordered_lengths = (
            ordered["_visible_days"]
            .astype(float)
            .tolist()
        )

        row["last_assignment_length"] = (
            ordered_lengths[-1]
            if ordered_lengths
            else np.nan
        )

        last3_lengths = ordered_lengths[-3:]

        row["mean_assignment_length_last3"] = (
            _safe_mean(last3_lengths)
        )

        row["std_assignment_length_last3"] = (
            _safe_std(last3_lengths)
        )

        row["assignment_length_trend"] = (
            _linear_trend(ordered_lengths)
        )

        historical_mean_assignment = (
            float(np.mean(ordered_lengths[:-1]))
            if len(ordered_lengths) > 1
            else np.nan
        )

        row["last_assignment_vs_historical_mean"] = (
            row["last_assignment_length"]
            - historical_mean_assignment
            if pd.notna(historical_mean_assignment)
            and pd.notna(row["last_assignment_length"])
            else np.nan
        )

        # Gap dynamics
        row["last_offwork_length"] = (
            float(gaps[-1])
            if gaps
            else 0.0
        )

        last3_gaps = gaps[-3:]

        row["mean_offwork_length_last3"] = (
            _safe_mean(last3_gaps)
            if last3_gaps
            else 0.0
        )

        row["std_offwork_length_last3"] = (
            _safe_std(last3_gaps)
            if last3_gaps
            else 0.0
        )

        row["offwork_gap_trend"] = (
            _linear_trend(gaps)
            if gaps
            else 0.0
        )

        historical_mean_gap = (
            float(np.mean(gaps[:-1]))
            if len(gaps) > 1
            else np.nan
        )

        row["last_gap_vs_historical_mean"] = (
            row["last_offwork_length"]
            - historical_mean_gap
            if pd.notna(historical_mean_gap)
            else np.nan
        )

        row["current_gap_ratio_to_historical"] = (
            row["days_since_last_assignment"]
            / historical_mean_gap
            if pd.notna(historical_mean_gap)
            and historical_mean_gap > 0
            else np.nan
        )

        # Recent rolling windows
        row.update(
            _calculate_recent_window_features(
                cg_assignments=cg_assignments,
                cutoff=cutoff,
                recruitment_date=recruitment_date,
                windows=recent_windows
            )
        )

        # ----------------------------------------------------
        # Client repetition / stability
        # ----------------------------------------------------

        client_sequence = ordered[
            assignment_client_col
        ].tolist()

        seen = set()
        repeat_count = 0

        for client in client_sequence:

            if client in seen:
                repeat_count += 1

            seen.add(client)

        row["repeat_client_ratio"] = (
            repeat_count / len(client_sequence)
            if client_sequence
            else 0.0
        )

        # ====================================================
        # Client-class exposure
        # ====================================================

        total_classified_days = 0

        class_days_dict = {}

        for cls in client_classes:

            cls_days = cg_assignments.loc[
                cg_assignments[
                    client_class_col
                ] == cls,
                "_visible_days"
            ].sum()

            cls_days = int(cls_days)

            class_days_dict[cls] = cls_days

            total_classified_days += cls_days

            row[
                f"days_client_class_{cls}"
            ] = cls_days

        for cls in client_classes:

            row[
                f"share_client_class_{cls}"
            ] = (
                class_days_dict[cls]
                /
                total_classified_days
                if total_classified_days > 0
                else 0.0
            )

        row["classified_client_days"] = (
            total_classified_days
        )

        # ----------------------------------------------------
        # Average client care intensity
        # ----------------------------------------------------

        numeric_classes = []

        for cls, days in class_days_dict.items():

            try:

                numeric_classes.append(
                    (float(cls), days)
                )

            except (TypeError, ValueError):

                pass

        if numeric_classes and total_classified_days:

            row["average_client_class"] = (
                sum(
                    cls * days
                    for cls, days
                    in numeric_classes
                )
                /
                total_classified_days
            )

        else:

            row["average_client_class"] = np.nan

        # ----------------------------------------------------
        # Current / previous client class
        # ----------------------------------------------------

        class_history = (
            ordered[
                [
                    "_visible_start",
                    client_class_col
                ]
            ]
            .dropna(
                subset=[client_class_col]
            )
        )

        if len(class_history) >= 1:

            row["current_client_class"] = (
                class_history.iloc[-1][
                    client_class_col
                ]
            )

        else:

            row["current_client_class"] = np.nan

        if len(class_history) >= 2:

            row["previous_client_class"] = (
                class_history.iloc[-2][
                    client_class_col
                ]
            )

            try:

                row["client_class_change"] = (
                    float(
                        row[
                            "current_client_class"
                        ]
                    )
                    -
                    float(
                        row[
                            "previous_client_class"
                        ]
                    )
                )

            except Exception:

                row["client_class_change"] = np.nan

        else:

            row["previous_client_class"] = np.nan
            row["client_class_change"] = np.nan

        # ====================================================
        # Caregiver-client fit
        # ====================================================

        # These features compare client requirements with caregiver
        # experience indicators using only assignments visible before
        # the snapshot.
        #
        # The contribution is day-weighted so longer assignments carry
        # proportionally more exposure.

        total_requirement_exposure_days = 0.0
        matched_requirement_days = 0.0
        unmatched_requirement_days = 0.0
        requirement_opportunity_days = 0.0

        requirement_types_seen = 0
        requirement_types_matched = 0

        mismatch_days_by_requirement = {}

        for requirement_col, caregiver_exp_col in (
            requirement_experience_map.items()
        ):

            if requirement_col not in cg_assignments.columns:
                continue

            if caregiver_exp_col not in caregiver.index:
                continue

            caregiver_has_experience = pd.to_numeric(
                pd.Series([caregiver[caregiver_exp_col]]),
                errors="coerce"
            ).iloc[0]

            caregiver_has_experience = (
                1
                if pd.notna(caregiver_has_experience)
                and float(caregiver_has_experience) > 0
                else 0
            )

            requirement_flag = pd.to_numeric(
                cg_assignments[requirement_col],
                errors="coerce"
            ).fillna(0)

            requirement_days = float(
                (
                    cg_assignments["_visible_days"]
                    *
                    (requirement_flag > 0).astype(int)
                ).sum()
            )

            if requirement_days <= 0:
                mismatch_days_by_requirement[
                    requirement_col
                ] = 0.0
                continue

            requirement_types_seen += 1
            total_requirement_exposure_days += requirement_days
            requirement_opportunity_days += requirement_days

            if caregiver_has_experience:
                matched_requirement_days += requirement_days
                requirement_types_matched += 1
                mismatch_days_by_requirement[
                    requirement_col
                ] = 0.0
            else:
                unmatched_requirement_days += requirement_days
                mismatch_days_by_requirement[
                    requirement_col
                ] = requirement_days

        row["requirement_exposure_days"] = (
            total_requirement_exposure_days
        )

        row["matched_requirement_days"] = (
            matched_requirement_days
        )

        row["unmatched_requirement_days"] = (
            unmatched_requirement_days
        )

        row["caregiver_client_match_ratio"] = (
            matched_requirement_days
            / requirement_opportunity_days
            if requirement_opportunity_days > 0
            else np.nan
        )

        row["caregiver_client_mismatch_ratio"] = (
            unmatched_requirement_days
            / requirement_opportunity_days
            if requirement_opportunity_days > 0
            else np.nan
        )

        row["requirement_type_match_ratio"] = (
            requirement_types_matched
            / requirement_types_seen
            if requirement_types_seen > 0
            else np.nan
        )

        row["requirement_types_seen"] = (
            requirement_types_seen
        )

        row["requirement_types_matched"] = (
            requirement_types_matched
        )

        # Optional per-requirement mismatch exposure
        for requirement_col, mismatch_days in (
            mismatch_days_by_requirement.items()
        ):

            safe_req = (
                requirement_col
                .replace(":", "")
                .replace(" ", "_")
                .replace("-", "_")
                .replace(".", "_")
                .replace("/", "_")
            )

            row[
                f"mismatch_days_{safe_req}"
            ] = mismatch_days

        # ====================================================
        # Package exposure
        # ====================================================

        for package in package_columns:

            safe_name = (
                package
                .replace("paket_", "")
                .replace(" ", "_")
                .replace("-", "_")
            )

            package_values = (
                pd.to_numeric(
                    cg_assignments[package],
                    errors="coerce"
                )
                .fillna(0)
            )

            days = (
                cg_assignments["_visible_days"]
                *
                package_values
            ).sum()

            row[
                f"days_package_{safe_name}"
            ] = days

            row[
                f"share_package_{safe_name}"
            ] = (
                days / workdays
                if workdays > 0
                else 0.0
            )

        result_rows.append(row)

    result = pd.DataFrame(result_rows)

    return result

### Cutoff modes\n
\n
The snapshot generator now supports several cutoff strategies:\n
\n
- `random_days`: **recommended main forecasting setup**. Draws one landmark between `min_days` and `max_days` after recruitment without using the future exit date.\n
- `days`: fixed landmark, e.g. 365 days after recruitment.\n
- `date`: fixed calendar prediction date.\n
- `ratio`: retrospective sensitivity analysis only because the cutoff depends on the eventual observed end.\n
- `full`: retrospective analysis only.\n
\n
For `random_days`, `days`, and `date`, caregivers who are no longer observable at the requested landmark are excluded rather than moving the cutoff backwards to their exit date. This prevents future-information leakage.\n

In [5]:
# ============================================================
# Optional caregiver-client requirement/experience mapping
# ============================================================
#
# Left side  = requirement in CLIENT data
# Right side = corresponding experience indicator in CAREGIVER data
#
# Extend or remove mappings depending on which columns are reliable
# in your final data.

requirement_experience_map = {
    "beginnende-demenz": "erfahrung-demenz-beginnend",
    "fortgeschrittene-demenz-alzheimer": "erfahrung-alzheimer",
    "aggressive-demenz": "erfahrung-demenz-aggressiv",
    "schlaganfall": "erfahrung-schlaganfall",
    "parkinson": "erfahrung-parkinson",
    "copd": "erfahrung-copd",
    "Klient:innen_Krebserkrankung_Ja": "erfahrung-krebs",
    "sub-injektionen-thrombose": "erfahrung-thrombose",
    "asthma": "erfahrung-asthma",
    "hilfsmittel-rollstuhl": "erfahrung-mobilitaet-rollstuhl",
    "bettlaegrig": "erfahrung-bettspflege",
    "harninkontinenz": "erfahrung-inko",
    "dauerkatheter": "erfahrung-dauerkatheter",
    "diabetes-medikamentoes": "erfahrung-diabetes",
    "diabetes-insulinpflichtig": "erfahrung-insulinpflichtig",
    "peg-sonde": "erfahrung-sondennahrung",
    "colostoma": "erfahrung-colostoma",
    "trachaeostoma": "erfahrung-tracheostoma",
    "hilfsmittel-sauerstoff": "erfahrung-o2",
    "Klient:innen_Allergien_Ja": "erfahrung-allergien",
    "depression": "erfahrung-depression",
    "palliativbetreuung": "erfahrung-palliativ",
}


In [6]:
# Recommended main experiment:
# one reproducible random prediction landmark per caregiver,
# sampled independently of the future exit date.
snapshot_df = build_caregiver_snapshot_dataset(
    caregivers=caregivers[caregivers["id"].isin(assignments["assignee"])],
    assignments=assignments,
    clients=clients,

    cut=CutConfig(
        mode="random_days",
        min_days=180,
        max_days=730,
        random_state=42
    ),

    study_end="2026-08-31",

    client_class_col="cluster",

    requirement_experience_map=requirement_experience_map,

    recent_windows=(90, 180, 365)
)

print(snapshot_df["snapshot_landmark_days"].describe())
print(snapshot_df["snapshot_mode"].value_counts())


count    3183.000000
mean      429.408106
std       159.404414
min       180.000000
25%       288.000000
50%       417.000000
75%       563.000000
max       730.000000
Name: snapshot_landmark_days, dtype: float64
snapshot_mode
random_days    3183
Name: count, dtype: int64


In [7]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class ModuleConfig:
    demographic: List[str] = field(default_factory=list)
    experience: List[str] = field(default_factory=list)
    organisational: List[str] = field(default_factory=list)

    include_work_history: bool = True
    include_client_exposure: bool = True
    include_assignment_stability: bool = True
    include_temporal_dynamics: bool = True
    include_caregiver_client_fit: bool = True
    include_package_exposure: bool = False


def build_feature_modules(
    snapshot_df: pd.DataFrame,
    config: ModuleConfig
):
    """
    Build interpretable predictor modules from the flattened caregiver snapshot.
    """

    modules = {}

    modules["demographic"] = [
        c for c in config.demographic
        if c in snapshot_df.columns
    ]

    modules["experience"] = [
        c for c in config.experience
        if c in snapshot_df.columns
    ]

    modules["organisational"] = [
        c for c in config.organisational
        if c in snapshot_df.columns
    ]

    # --------------------------------------------------------
    # Work history
    # --------------------------------------------------------
    if config.include_work_history:
        candidates = [
            "days_since_recruitment",
            "number_assignments",
            "number_unique_clients",
            "workdays",
            "offworkdays",
            "work_ratio",
            "mean_assignment_length",
            "median_assignment_length",
            "min_assignment_length",
            "max_assignment_length",
            "std_assignment_length",
            "mean_offwork_length",
            "median_offwork_length",
            "max_offwork_length",
            "days_since_last_assignment",
        ]

        modules["work_history"] = [
            c for c in candidates
            if c in snapshot_df.columns
        ]
    else:
        modules["work_history"] = []

    # --------------------------------------------------------
    # Client exposure
    # --------------------------------------------------------
    if config.include_client_exposure:
        modules["client_exposure"] = [
            c for c in snapshot_df.columns
            if (
                c.startswith("days_client_class_")
                or c.startswith("share_client_class_")
                or c in [
                    "average_client_class",
                    "classified_client_days",
                ]
            )
        ]
    else:
        modules["client_exposure"] = []

    # --------------------------------------------------------
    # Assignment stability
    # --------------------------------------------------------
    if config.include_assignment_stability:
        candidates = [
            "repeat_client_ratio",
            "current_client_class",
            "previous_client_class",
            "client_class_change",
        ]

        modules["assignment_stability"] = [
            c for c in candidates
            if c in snapshot_df.columns
        ]
    else:
        modules["assignment_stability"] = []

    # --------------------------------------------------------
    # Temporal dynamics
    # --------------------------------------------------------
    if config.include_temporal_dynamics:
        explicit_temporal = [
            "last_assignment_length",
            "mean_assignment_length_last3",
            "std_assignment_length_last3",
            "assignment_length_trend",
            "last_assignment_vs_historical_mean",
            "last_offwork_length",
            "mean_offwork_length_last3",
            "std_offwork_length_last3",
            "offwork_gap_trend",
            "last_gap_vs_historical_mean",
            "current_gap_ratio_to_historical",
        ]

        rolling_temporal = [
            c for c in snapshot_df.columns
            if (
                c.startswith("workdays_last_")
                or c.startswith("offworkdays_last_")
                or c.startswith("work_ratio_last_")
                or c.startswith("assignments_last_")
            )
        ]

        modules["temporal_dynamics"] = [
            c for c in dict.fromkeys(
                explicit_temporal + rolling_temporal
            )
            if c in snapshot_df.columns
        ]
    else:
        modules["temporal_dynamics"] = []

    # --------------------------------------------------------
    # Caregiver-client fit
    # --------------------------------------------------------
    if config.include_caregiver_client_fit:
        explicit_fit = [
            "requirement_exposure_days",
            "matched_requirement_days",
            "unmatched_requirement_days",
            "caregiver_client_match_ratio",
            "caregiver_client_mismatch_ratio",
            "requirement_type_match_ratio",
            "requirement_types_seen",
            "requirement_types_matched",
        ]

        mismatch_features = [
            c for c in snapshot_df.columns
            if c.startswith("mismatch_days_")
        ]

        modules["caregiver_client_fit"] = [
            c for c in dict.fromkeys(
                explicit_fit + mismatch_features
            )
            if c in snapshot_df.columns
        ]
    else:
        modules["caregiver_client_fit"] = []

    # --------------------------------------------------------
    # Optional package exposure
    # --------------------------------------------------------
    if config.include_package_exposure:
        modules["package_exposure"] = [
            c for c in snapshot_df.columns
            if (
                c.startswith("days_package_")
                or c.startswith("share_package_")
            )
        ]
    else:
        modules["package_exposure"] = []

    # Complete model = union of all active modules
    complete = []
    for name, cols in modules.items():
        if name == "complete":
            continue
        complete.extend(cols)

    modules["complete"] = list(dict.fromkeys(complete))

    return modules


# Example module configuration.
# Adjust the explicit static feature lists to the columns you want to retain.
module_config = ModuleConfig(
    demographic=[
        "age_at_snapshot",
        "gender_M",
        "nationality_Rumänien",
        "nationality_Ungarn",
        "nationality_Slowakei",
        "nationality_Polen",
        "nationality_Österreich",
    ],

    experience=[
        "ausbildung-krankenschwester",
        "erfahrung-krankenhaus",
        "erfahrung-alzheimer",
        "erfahrung-demenz-beginnend",
        "erfahrung-demenz-aggressiv",
        "erfahrung-schlaganfall",
        "erfahrung-parkinson",
        "erfahrung-copd",
        "erfahrung-mobilitaet-rollstuhl",
        "erfahrung-bettspflege",
        "erfahrung-diabetes",
        "erfahrung-palliativ",
    ],

    organisational=[
        "geschBereich_Geschäftsbereich H24",
        "geschBereich_Geschäftsbereich PML",
    ],

    include_work_history=True,
    include_client_exposure=True,
    include_assignment_stability=True,
    include_temporal_dynamics=True,
    include_caregiver_client_fit=True,
    include_package_exposure=False,
)

modules = build_feature_modules(
    snapshot_df,
    module_config
)

for module_name, feature_names in modules.items():
    print(
        f"{module_name:25s}: "
        f"{len(feature_names):3d} features"
    )


demographic              :   7 features
experience               :  12 features
organisational           :   2 features
work_history             :  15 features
client_exposure          :  10 features
assignment_stability     :   4 features
temporal_dynamics        :  23 features
caregiver_client_fit     :  30 features
package_exposure         :   0 features
complete                 : 103 features


In [9]:
snapshot_df.to_csv("snapshot.csv", sep=";")